# TDK – ECG Age / AgeGap baseline
**Téma:** Biológiai életkor és egészségi állapot becslése 12 elvezetéses EKG-jelek mélytanulási elemzésével

Ez a notebook a mellékelt Python modulokat használja. Először állítsd be a `DATA_DIR` útvonalat a PTB-XL v1.0.3 gyökérkönyvtárára.

In [ ]:
from pathlib import Path
DATA_DIR = Path('/UTVONAL/PTBXL')
RUN_DIR = Path('runs/baseline')
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR


## 1. Metaadatok ellenőrzése
Csak a 18–89 éves rekordokat tartjuk meg. A hivatalos `strat_fold` alapján 1–8 = train, 9 = validation, 10 = test.

In [ ]:
from ecg_age_common import load_ptbxl_metadata
db = load_ptbxl_metadata(DATA_DIR, 18, 89)
print(db.shape)
display(db[['patient_id','age','sex','strat_fold','filename_lr']].head())
display(db.groupby('strat_fold').size())


## 2. Modell tanítása
Az első baseline egy 1D ResNet. A fő mérőszám a validation MAE; a legjobb checkpoint automatikusan mentésre kerül.

In [ ]:
!python train_ecg_age.py --data-dir "{DATA_DIR}" --epochs 30 --batch-size 64 --output-dir "{RUN_DIR}"


## 3. Tanítási eredmények


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
metrics = json.load(open(RUN_DIR/'metrics.json'))
metrics


In [ ]:
hist = pd.read_csv(RUN_DIR/'training_history.csv')
display(hist.tail())
plt.figure(figsize=(7,4))
plt.plot(hist['epoch'], hist['mae'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Validation MAE (év)')
plt.title('Életkorbecslő modell validációs MAE')
plt.show()


## 4. AgeGap és bias-korrekció
A korrekció paramétereit csak a validation halmazon illesztjük, majd változtatás nélkül alkalmazzuk a test halmazra.

In [ ]:
!python evaluate_agegap.py --data-dir "{DATA_DIR}" --predictions "{RUN_DIR/'predictions.csv'}" --output-dir "{RUN_DIR/'analysis'}"


In [ ]:
agegap = pd.read_csv(RUN_DIR/'analysis/test_agegap_with_labels.csv')
display(agegap.head())
display(pd.read_csv(RUN_DIR/'analysis/diagnostic_group_agegap.csv'))


## 5. XAI – Integrated Gradients
Válassz egy teszt EKG-t és futtasd az interpretálhatósági elemzést.

In [ ]:
test_id = int(agegap.iloc[0]['ecg_id'])
!python explain_integrated_gradients.py --data-dir "{DATA_DIR}" --checkpoint "{RUN_DIR/'best_model.pt'}" --ecg-id {test_id} --output-dir "{RUN_DIR/'xai'}"


## 6. TDK-ban riportálandó eredmények
- test MAE, RMSE, R² és Pearson-r
- predicted age vs. chronological age scatterplot
- nyers és korrigált AgeGap életkorfüggése
- AgeGap NORM/MI/STTC/CD/HYP csoportokban
- logisztikus regressziós OR / +5 év AgeGap
- Integrated Gradients alapján legfontosabb elvezetések és jelrészletek
